# Lektion 13 - Agentenspeicher


## Einrichtung

Dieses Notebook zeigt, wie man einen Reisebuchungsagenten mit **persistierendem Speicher** mithilfe des **Microsoft Agent Framework** (MAF) erstellt.

Sie erfahren, wie sich verschiedene Arten von Agentenspeicher — Arbeits-, Kurzzeit- und Langzeitspeicher — darauf auswirken, wie ein Agent Informationen über Gespräche hinweg speichert und verwendet.

**Voraussetzungen:**
- Ein Azure AI Foundry-Projekt mit einem bereitgestellten Chatmodell (z.B. `gpt-4o-mini`).
- Eingeloggt mit der Azure CLI — führen Sie `az login` in Ihrem Terminal aus.
- `AZURE_AI_PROJECT_ENDPOINT` — der Endpunkt Ihres Azure AI Foundry-Projekts.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — der Name Ihres bereitgestellten Modells.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Arten von Agentenspeicher

KI-Agenten können verschiedene Arten von Speicher nutzen, die jeweils einem bestimmten Zweck dienen:

### Arbeitsgedächtnis
Der Gesprächsverlauf selbst — die Nachrichten, die in einer einzigen Sitzung ausgetauscht werden. Der Agent kann auf frühere Nachrichten im selben Verlauf zurückgreifen, um Kohärenz zu bewahren. In MAF wird dies über **`agent.create_session()`** erstellt, welches eine `AgentSession` zurückgibt.

### Kurzzeitgedächtnis
Informationen, die für die Dauer einer Aufgabe oder Sitzung bestehen bleiben, aber nicht dauerhaft gespeichert werden. Zum Beispiel kann der Agent während eines mehrstufigen Planungsgesprächs Fakten sammeln und diese nutzen, um eine endgültige Reiseroute zu erstellen.

### Langzeitgedächtnis
Präferenzen und Fakten, die **sitzungsübergreifend** erhalten bleiben. Ein zurückkehrender Nutzer sollte seine Diätbeschränkungen oder Reisepräferenzen nicht erneut angeben müssen. Langzeitgedächtnis wird typischerweise von einem externen Speicher unterstützt — einer Datenbank, Datei oder einem Vektorindex — und dem Agenten über Tools bereitgestellt.


## Arbeitsgedächtnis mit Sitzungen

Die einfachste Form des Gedächtnisses ist die Gesprächssitzung. Wenn Sie dieselbe Sitzung (erstellt über `agent.create_session()`) an aufeinanderfolgende `agent.run()`-Aufrufe übergeben, sieht der Agent die vollständige Historie dieses Gesprächs und kann sich an frühere Details erinnern.

Lassen Sie uns einen Reiseagenten erstellen und das Arbeitsgedächtnis demonstrieren.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Der Agent erinnerte sich korrekt an das Budget, weil beide Nachrichten dieselbe Sitzung teilen. Dies ist **Arbeitsgedächtnis** — es existiert nur für die Dauer der Sitzung.

### Was passiert bei einem neuen Thread?

Wenn wir eine **neue** Sitzung erstellen, hat der Agent kein Gedächtnis an das vorherige Gespräch:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Langzeitgedächtnis-Muster

Um sich an Benutzerpräferenzen **über Sitzungen hinweg** zu erinnern, benötigen wir einen dauerhaften Speicher, der außerhalb des Gesprächsverlaufs lebt. Der Agent greift über **Tools** auf diesen Speicher zu — Funktionen, die er zum Speichern und Abrufen von Informationen aufrufen kann.

Unten implementieren wir einen einfachen Präferenzspeicher im Speicher (in der Produktion würden Sie diesen mit einer Datenbank oder einem Vektorindex unterstützen) und stellen ihn als Tools bereit, die der Agent verwenden kann.

### Architektur
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Szenario 1 — Erstmaliger Nutzer bucht eine Jubiläumsreise

Sarah besucht die Seite zum ersten Mal. Der Agent sollte ihre Vorlieben über die Tools speichern und diese nutzen, um Hotels zu empfehlen.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Szenario 2 — Sarah kehrt Wochen später zurück

Sarah startet einen **ganz neuen Thread** (simuliert eine neue Sitzung). Das Arbeitsgedächtnis ist leer, aber der Langzeitpräferenzspeicher enthält noch ihre Informationen. Der Agent sollte diese abrufen und nutzen, um Empfehlungen zu personalisieren.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Zusammenfassung

In dieser Lektion haben Sie drei Arten von Agentenspeicher kennengelernt und wie man sie mit dem Microsoft Agent Framework implementiert:

| Speichertyp | MAF-Mechanismus | Lebensdauer |
|---|---|---|
| **Arbeits-** | `agent.create_session()` | Einzelne Unterhaltung |
| **Kurzzeit-** | Kumulierte Kontextinformationen innerhalb eines Threads | Einzelne Aufgabe / Sitzung |
| **Langzeit-** | Externer Speicher, zugänglich über `@tool`-Funktionen | Über mehrere Sitzungen hinweg |

### Wichtige Erkenntnisse
1. **`agent.create_session()`** bietet Arbeitsgedächtnis — der Agent sieht den gesamten Gesprächsverlauf innerhalb einer Sitzung.
2. **Neue Sitzungen verlieren den Kontext** — ohne Langzeitgedächtnis kann sich der Agent nicht an frühere Gespräche erinnern.
3. **`@tool`-Funktionen** überbrücken die Lücke — sie ermöglichen dem Agenten, Informationen in einem persistenten Speicher zu sichern und abzurufen.
4. **Personalisierung verbessert sich mit der Zeit** — je mehr Präferenzen gespeichert werden, desto besser sind die Empfehlungen des Agenten.

### Praxisbeispiele
- **Kundendienst**: Kundenhistorie und Vorlieben merken
- **Persönliche Assistenten**: Kontext über Tage oder Wochen beibehalten
- **Gesundheitswesen**: Patienteninformationen und Präferenzen verfolgen
- **E-Commerce**: Personalisierte Einkäufe basierend auf der Historie

### Nächste Schritte
- Ersetzen des In-Memory-Dictionary durch eine Datenbank oder einen Vektorspeicher (z. B. Azure AI Search)
- Einführung einer Ablauffrist für zeitkritische Informationen
- Aufbau von Multi-Agenten-Systemen mit gemeinsamem Speicher
- Erkundung des Cognee-Notebooks für speicherbasierte Wissensgraphen


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:  
Dieses Dokument wurde mithilfe des KI-Übersetzungsdienstes [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir um Genauigkeit bemüht sind, können automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten. Das Originaldokument in seiner Ursprungssprache ist als maßgebliche Quelle zu betrachten. Für kritische Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Nutzung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
